## Building a Simplpe LLM Application with LCEL
In this quickstart we'll show you how to bulid a simple LLM application with langchain. This application will traslate text from English into another language. This is a relative simple Application-it's just a single LLm call plus some prompting Still, this is a great way to get start wioth langchain- a lot of feature can be build with just some prompting and an LLM call!
After seeing this video, You'll have a high level overview of:

* Using Language models 
* Using promptTemplets and outputParser
* Using Langchain Expression language (LCEL) to chain component together
* Debugging and tracing Our Application using langSmith
* deploying Our application with langserve


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
import openai
openai_api_key=os.getenv("OPENAI_API_KEY")
groq_api_key=os.getenv("GROQ_API_KEY")



In [18]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
model1 = ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)
model1

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000244BB345DB0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000244BB404340>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
from langchain_core.messages import HumanMessage,SystemMessage

messages1=[
    SystemMessage(content="Translate the following from English to French"),
    HumanMessage(content="Hekko How are you?")
]
response=model1.invoke(messages1)


AIMessage(content='La réponse en français serait :\n\n"Bonjour, je vais bien, merci."\n\nSi vous voulez une réponse plus informelle, vous pouvez utiliser :\n\n"Ça va bien, merci."', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 49, 'total_tokens': 89, 'completion_time': 0.071650635, 'completion_tokens_details': None, 'prompt_time': 0.002812231, 'prompt_tokens_details': None, 'queue_time': 0.053391888, 'total_time': 0.074462866}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019de8aa-39da-7380-8b5f-b3bc70a7eee8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 40, 'total_tokens': 89})

In [20]:
response

AIMessage(content='Bonjour, comment allez-vous ?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 48, 'total_tokens': 56, 'completion_time': 0.016770307, 'completion_tokens_details': None, 'prompt_time': 0.002696193, 'prompt_tokens_details': None, 'queue_time': 0.051378907, 'total_time': 0.0194665}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019de8a9-4d6c-7c53-9906-5d80279a7163-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 8, 'total_tokens': 56})

In [21]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(response)

'Bonjour, comment allez-vous ?'

In [22]:
### Using LCEL - chain the components\

chain=model1|parser
chain.invoke(messages1)

'"Bonjour, comment allez-vous ?"'

In [23]:
## prompt Tem0plets
from langchain_core.prompts import ChatPromptTemplate
generic_template="Translate the following into{language}:"
prompt=ChatPromptTemplate.from_messages(
    [
        ("system",generic_template),
        ("user","{text}")
    ]
)

In [26]:
result=prompt.invoke({"language":"French","text":"hello"})

In [27]:
result.to_messages()

[SystemMessage(content='Translate the following intoFrench:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='hello', additional_kwargs={}, response_metadata={})]

In [29]:
## Chaining together component with LCEl
chain=prompt|model1|parser
chain.invoke({"language":"French","text":"hello"})

'Bonjour.'